### 1. Fetch credentials securely

In [ ]:
SCOPE_NAME = "hospital-scope"
EVENT_HUB_CONN_STRING = dbutils.secrets.get(scope=SCOPE_NAME, key="event-hub-connection-string")
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope=SCOPE_NAME, key="storage-account-key")

EVENT_HUB_NAMESPACE = "mha-hospital-analytics-namespace.servicebus.windows.net:9093"
EVENT_HUB_NAME = "hospital-analytics-eh"
STORAGE_ACCOUNT_NAME = "mhahospitalstorage"
BRONZE_CONTAINER = "bronze"

### 2. Use the Databricks shaded class path for SASL authentication

In [ ]:
jaas_rule = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EVENT_HUB_CONN_STRING}";'

kafka_options = {
    "kafka.bootstrap.servers": EVENT_HUB_NAMESPACE,
    "subscribe": EVENT_HUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": jaas_rule
}

### 3. Read stream from Event Hubs

In [ ]:
raw_df = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

json_df = raw_df.selectExpr("CAST(value AS STRING) as raw_json")

### 4. Set ADLS Gen2 key and write to Delta

In [ ]:
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

bronze_path = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/patient_flow_raw"
checkpoint_path = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/_checkpoints/patient_flow_raw"

query = (json_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .start(bronze_path)
)